# MediaVault · серверный импорт в Google Drive

Видео скачивается **на сервер Google Colab**, собирается в MP4, шифруется AES-GCM
и загружается в ваш Drive. Телефон передаёт небольшое задание; видеотрафик при
импорте не идёт через телефон, GitHub Pages или Cloudflare Worker.

1. На сайте MediaVault откройте свою библиотеку → **Импорт**. Укажите публичные
   ссылки, коллекцию и серии, нажмите **Подготовить серверный импорт → Скопировать задание**.
2. Здесь выберите **Среда выполнения → Выполнить всё**. Достаточно бесплатного CPU;
   GPU, Pro, банковская карта и Cloudflare не требуются.
3. Вставьте скопированное задание в появившееся поле. Затем войдите в аккаунт
   владельца и введите фразу библиотеки в скрытом поле `getpass`.
4. После завершения нажмите **Обновить библиотеку** на сайте.

Используется собственный разбор публичного HTML и открытые адаптеры плееров
`anime-dl-core`. API аниме-сайтов, DRM и платный доступ не используются.
Поддерживаются найденные публичные MP4/HLS и плееры Kodik, Aniboom, Sibnet,
Animedia. Если сайт отдаёт только проверку браузера или скрывает список серий,
понадобятся отдельные публичные ссылки; поддержка всех зеркал не гарантируется.

Бесплатные сессии Colab имеют переменную длительность и могут отключаться.
После каждой завершённой серии результат сохраняется в Drive. Повторный запуск
того же задания пропустит уже записанные серии. Временные файлы занимают место
только для текущей серии и удаляются после неё. Пока идёт импорт, не изменяйте
библиотеку с другого устройства. В Drive должно хватать свободного места.

**Не вставляйте пароль Google, Client Secret, токены или секретную фразу в JSON
задания или ячейки кода.** Фраза вводится только в скрытом поле при запуске и
не записывается в блокнот. Не делитесь работающей сессией и её выводом.

In [ ]:
# 1. Подготовка среды. Код загружается и проверяется до входа в Google.
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, urllib.request

SOURCE_HASHES = {'server/catalog_taxonomy.py': '5ac9bd90dab4e171a8cd23d2fecd89efe98a7f67b1b7c7f3723d743ea55c8fbf', 'server/catalog-taxonomy.json': '4033c2198593cf42868dd7fc4d24593c4e0b08317d6deb21dd18b30dae8cbf62', 'server/source_parser.py': '646df6a5543621ded9fd5d6b276cf50cb723162252e9c1947fef6710aa53d289', 'server/media_download.py': '665d5c6f46dc013dec616b15a65140c99fd03964f85a820c555014c8c7e8f8e9', 'server/run_import.py': '92125208479c3325a6ebb7ab0e43e9bb60306734661cef7783069f1db097f956', 'server/vault_drive.py': 'd76585bff78c62402ed9f3e100097f6db11b1a1d211eac90d8aa3f5fb42f29d6', 'server/player_catalog.py': '90a909760d591f4c368b0da801eba277a30c19fab458e261df1ddfda1fb34c18', 'server/player_compat.py': 'bd18fd66037d7bc9e2f30eaa34553d320f7477e07758b1a05c4e5da3766df3bc', 'server/browser_player.py': '6cbf9710385bf5b5a92e199db6eac4a6a6f43793268835530df7af3a5a8285f5', 'server/public_page_data.py': '9c94065d91ee3f418b19f9de248abacf64dda79fb2414879d15ea72aa9998fa0', 'server/requirements.txt': 'c5ea33be7878586d3e27dbfcff4aa6e6fa37b0d9beebcadaf052b7c8993be009', 'server/title_extras.py': '679bb5169223e64051163ceb14bb533ecfa7363c41dbfd6a8804fc0b02cfd743', 'server/manga_sources.py': '36dd3681d53409b96e326c287ff2de1b9e0627194b102ba3feca9b4ada8a54ae', 'server/manga_novelcool.py': '744b7c5262b9b8169e1f00cc929f84aca6acdbdcd8d87eedd03ac02556e3dd20', 'server/novelcool_client.py': 'ba2faefe7ea37312954962f06fc9d84a283bda96d827c6bb8a16296c388bfd87', 'server/manga_raw_adapters.py': 'e535e249aed7a35c6d91660321ce9bd1976725ee2278dd81800fb711afe54c34', 'server/manga_webtoons.py': 'e800693d1c28ea7fedf017dde3c2a9c981cb45a50dc73fa3608c58c85647d743', 'server/manga_catalog_adapters.py': 'af61800ff175b4b75687e672a057473348575f2fce9d93ffd1856ebf3fcd1e54', 'server/manga_download.py': 'd4ff59410781460ed817f8a31c896317db3ccb0b2f35369a1804b74baea42f80', 'server/title_identity.py': '218d0761ac0536ff58f758fcd0e31bda578ddb7de4cb2b4bc54e3e78f2d95c10', 'server/suwayomi_bridge.py': 'a16e07a765b89ffe7c53fd37f45146603d689bb9b93532fccb2e38802bf0eb80', 'server/manga_solver.py': 'cd91830093ac7192e2556e6ad5ee29d678fe7f3f8787be542c6018790819a5d2', 'server/manga_hmanga.py': '6f2d4e717fe535b1c6dfefe7a0f6190460ed6c58867db888f520894ba97c0beb', 'server/manga_source_tags.py': '5793dce9272209eb4a63b93afe9ad503c4d6c238eb373061e64af0c89c990144', 'server/manga_source_coverage.py': '44c496d93b3c75eefbfbd49ce85c203788537bc3285e24cc8b8983e3b05a3750', 'server/manga_catalog_index.py': '570275803e4dac75cc7a5bd3807ea52817bfa6e96b79e01776e8171eee2394c9', 'server/manga_tag_policy.py': 'a1e2c22750459902819ea7f684ce023675a599c8b542c53cf5cc0ed3ffb03ad4', 'server/manga_tag_deploy.py': '273f4b2e6b9b5fabbd9bd55a844733f0acd776b201fb2493d43461e2f941217c', 'server/manga_catalog_metadata.py': '73ffe6274e77812395244a4e8a49e482e350486031554619cc556cb76d85ae36', 'server/content_catalog.py': '6e334bf974afc6299e3fd1f353297b006abe8a7991f2cb221c2349cdd12e714c', 'server/content_management.py': '7e30dd9df5042e2f0c84028e8d71b2527b53d51da8c8b5b3fcbb2c829ddd9583', 'server/content_sources.py': '0aca5f1c997110189773bedaa19a943347112d2dd214f15c0d0f05776bae6de8', 'server/content_metadata.py': '59d0966d0297b0f580b7dfcf3346a8b5fb4a07270cde8b3443f7e257d2b55ff8', 'server/content_source_routes.py': 'b89020a80dc700379bf77c31193907adf1237ec6d1052037e15af6b1ebd0718b', 'server/content_vocabulary.py': 'ce907aa4e2405c0c7ad0edc3f7ab0c01ea4153065b1bfb75abe0256e85073f8e', 'server/catalogue_sources.py': 'ec776856c6e9c22c49b423e05f2c5d5451dd7f069e038513c6a398ce8fb269d3', 'server/catalogue_html.py': '800d26cf0ed3db82b96acf26757e4ba9de194f57497789948f1bf89e9b63b181', 'server/catalogue_lib_auth.py': 'fda26317aef01742b252d95cc8d1a1d03a8e65afb74c421f4de99bb9a404e8b0', 'server/catalogue_indexer.py': 'b1363e4c4a63a097106c5d17ed5fc8f3fe16177f44fdb749503e372bf4f7174d', 'server/manga_native_sources.py': '9f47696dc76888e64e26a6fa7d11abd9f00293c80017597f897b6d35a5350b03', 'server/manga_mangashi.py': 'edca684687c2ec62037aa3f037bcea21b46efbdbc04aa505896631f089fa1072', 'server/manga_mangahub.py': 'ababc02924c2313f8db4f93c39d4b73d28fc3b5a0baae165f83fea8fe8ea500b', 'server/manga_madara.py': '6081f9f96de0b28d5f6b83cad3b12b74be3e258aca8cb3abf2b9d4f85b44c889', 'server/manga_madara_profiles.py': '46e1aefc9294a230700b6e4c5ea3cf2f716fd30202a815f2475a9289e17b355b', 'server/manga_source_urls.py': '04614a55da4f8bc47a3207fc37742946dfb153a51e647ae69ef01ba70ef4cef5', 'server/manga_themesia.py': 'b8a0217f24cce5895c8fded8af1d97741b63e1399c7a5788b87391694ada55e4', 'server/manga_iken.py': 'ebfdb2d7b7be0c814476ffefc8aa5914e34d905e3ce68cf4aad6a8d19bcac0b3', 'server/manga_mangacatalog.py': '0c571c7654b3606ed6028ad6cc5f1f62a458ce86c0028e8eec86d321e91149f2', 'server/manga_keyoapp.py': '44969d42b5633bc30a49e00ca79618539196266dce2f9a3b7e17b20bbca72974', 'server/manga_vinetheme.py': '3b0d03767e56cf67f13f29052276bbe01e4e1812d59cf30ccf7e503902099a1b', 'server/manga_hentaihand.py': 'ff4ff412ab68d896e0b938698bc69b561829aaea98b8c039f081cf838c4d30d4', 'server/manga_comici.py': '1ff9197d4e244b5be1dee956c5ce043057c6d94274cc4590c6458833e6686d47', 'server/manga_network_routes.py': '1aabb31685b2c6318ce5691891dabb6b982cd3e0f5bef02dbf9487258a28c880', 'server/manga_page_media.py': '41da29f983162317a0ed60de54e0f56bbc6c793589ca2a207e3ea7ca9a1e317e', 'server/manga_gigaviewer.py': '2180c40072eace1224fdb48afb6cdd0252bd5b0b6026db9fead5d8341af59336', 'server/manga_gigaviewer_profiles.py': '38d1a3897179403da1f9fccb15c0c5af73b85320b3f886e6e320651b81ba5fb1', 'server/manga_guya.py': '4206b6f17a4cf07b4aa64c3e0296779232352426ad1ded380c742c948549030c', 'server/manga_mangadventure.py': 'f79e3204585291512dc9027e31866f22d93212482c6452790f66e1f181aae4be', 'server/manga_mangak.py': '226a427455a2837a7d6ec449a90ba9877d99f6a42c0f45c034008e6016013b63', 'server/manga_mangadex_native.py': '6a4cda1166863521c1bb7087540804aa93ffec056905047e42a8cfccf6f5d348', 'server/manga_mangamillion.py': 'd782071e33b18d65e7f890df72ffaab5b6bb7e5d5e83415351ec9d7391fbaea7', 'server/manga_million_wire.py': '33a4c5c5162a22d4d609a967f9ea2ed3ebf81b9dce52a286e37d972c9b7012b1', 'server/manga_ball_million_profiles.py': '0f4ddfe0413d7f14f1dca78fdda3eb21461f820186f53d76bd115d14a85a3e17', 'server/manga_mangaball_native.py': 'c740a34d986ea6e7590902a81cbcdc0b291c67829d8d837f7ff9f585a53d7950', 'server/mangaball_reader.py': '93934bb97fdc265412c0ab81b8fbf65b9004cb768bfcaecb3ab603ce017ab729', 'server/manga_foolslide.py': '5cfc01172e7de3f631ae24c3699c2fe504e45d3185c71ebc46a6224ab65dae70', 'server/manga_dot_nami_http.py': 'a7baa2d1defc939bad19b5379339bbfab72fbe6753888eae620dc0841ae980ba', 'server/manga_dot_nami_profiles.py': 'f6d4c0df8f311d1e15eb87c275bfa33bd2ed3e53802f9c9fe9f37ef8fc98287e', 'server/manga_namicomi.py': '32a3a6d31f52d26b4c576ef08e03045934eb13fcdc716eaf0d99ececc8baf83c', 'server/manga_global_dbm_profiles.py': '4ebf8780b2896c2679e045e667e17001898e96fabf086168aac692e92196179d', 'server/manga_globalcomix.py': '19dfb84538e25483c16d4531d2d6198aa158db27c3c34a8b15d8bef9a7e7234a', 'server/manga_dbmultiverse.py': 'c0dc5d5dc86f74d3904f5e84e4206c85a828a75e37a287931b4a4e013cd0ed02', 'server/manga_xcomic_fury_profiles.py': 'b5bbb86ce923b287f3159edd13e9870d08aab579533d166fb0b9ad7e1682b7f0', 'server/manga_xcomic_queries.py': '8ba74937174fb0ad4219146e58d454e34a8982a37a817406862c891f74539bc1', 'server/manga_xcomic.py': '3e70289019709c71cff31eaaaf34523da84455813203d2be670813e061e229b3', 'server/manga_comicfury.py': 'd70bb32f9a198cc53a6f253f344867968e615743838069727cc0c26d39ac3082'}
CONFIG_FILES = ['server/manga-tag-policy.json']
BASE = "https://raw.githubusercontent.com/Onikri/MediaVault/main/"
RUNTIME = Path("/content/mediavault-server")
RUNTIME.mkdir(parents=True, exist_ok=True)
for relative, expected in SOURCE_HASHES.items():
    with urllib.request.urlopen(BASE + relative, timeout=45) as response:
        data = response.read(2 * 1024 * 1024)
    if hashlib.sha256(data).hexdigest() != expected:
        raise RuntimeError("Исходный код обновился. Откройте свежий блокнот по ссылке с сайта.")
    (RUNTIME / Path(relative).name).write_bytes(data)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r", str(RUNTIME / "requirements.txt")], check=True)
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
sys.path.insert(0, str(RUNTIME))
from manga_tag_policy import Policy
for relative in CONFIG_FILES:
    with urllib.request.urlopen(BASE + relative, timeout=45) as response:
        data = response.read(1_000_001)
    if len(data) > 1_000_000:
        raise RuntimeError("Слишком большой файл правил тегов.")
    Policy(json.loads(data))  # Mutable data only; executable sources remain hash-checked.
    (RUNTIME / Path(relative).name).write_bytes(data)
from run_import import validate_job, run_job
from vault_drive import GoogleDriveAdapter, VaultDrive, VaultError
from source_parser import SourceError
print("Среда готова. Платные ресурсы не включались.")

In [ ]:
# 2. Вставь задание кнопкой «Скопировать задание» на сайте (оно копируется одной строкой).
# Здесь нужны только ссылки и ID каталога. Фразу библиотеки сюда вводить нельзя.
job_text = input("Задание MediaVault (JSON): ").strip()
try:
    job = json.loads(job_text)
    validate_job(job)
except Exception:
    job_text = ""
    raise RuntimeError("Некорректное задание. Скопируйте его кнопкой на сайте заново.") from None
job_text = ""
print("Задание проверено. Далее откроется вход в Google.")

In [ ]:
# 3. Вход владельца, открытие библиотеки и серверный импорт.
from google.colab import auth
from getpass import getpass

auth.authenticate_user()
vault = None
phrase = None
try:
    # Адрес Google-аккаунта проверяется до любых записей в Drive.
    vault = VaultDrive(GoogleDriveAdapter.from_adc())
    phrase = getpass("Секретная фраза открытой библиотеки: ")
    try:
        vault.unlock(job["library_id"], phrase)
    except Exception:
        raise VaultError("Не удалось открыть библиотеку. Проверьте ID, аккаунт и секретную фразу.") from None
    finally:
        phrase = None
    result = run_job(job, vault)
    print("Откройте сайт и нажмите «Обновить библиотеку»:")
    print("https://onikri.github.io/MediaVault/?library=" + vault.library_id)
except (SourceError, VaultError) as error:
    print("Импорт остановлен:", str(error))
    print("Завершённые серии сохранены. После исправления причины можно запустить то же задание.")
except KeyboardInterrupt:
    print("Импорт прерван. Завершённые серии сохранены; то же задание можно продолжить.")
except Exception:
    print("Импорт остановлен из-за ошибки сети, доступа Google или квоты.")
    print("Проверьте свободное место и аккаунт. Повторите запуск; готовые серии будут пропущены.")
finally:
    phrase = None
    if vault is not None:
        vault.close()
    vault = None

### Что сохраняется

Drive содержит каталог, зашифрованные блоки по 4 МиБ и зашифрованные манифесты
восстановления. Разделы называются «Аниме», «Манга», «Ранобэ», «Книги», «Музыка»; имена тайтлов и
видео находятся внутри зашифрованного каталога, а папки тайтлов, сезонов/томов и файлов имеют случайные ID.
Совместимость с веб-приложением проверяется тестами Python ↔ Web Crypto.

Фраза и производный ключ используются в памяти текущего процесса. Авторизацией
Google управляет Colab. Завершите сессию через **Среда выполнения → Отключиться
и удалить среду выполнения**, когда импорт закончен. Во время обычного просмотра
на сайте видео, разумеется, скачивается из Drive на устройство зрителя.

[Исходный код и инструкции](https://github.com/Onikri/MediaVault)
· [Использованные открытые библиотеки](https://github.com/Onikri/MediaVault/blob/main/THIRD_PARTY.md)